# Validation design: контрольный корпус и стратификация

## Зачем нужен отдельный manifest

Recall@50 легко завысить или занизить неудачной выборкой запросов. Например, запросы без фильтров и запросы из маленьких городов имеют разную сложность, а location-channel особенно силён там, где локальный пул мал. Поэтому все BM25, dense, RRF и SPLADE эксперименты должны использовать **одни и те же query rows и relevance sets**.

В train нет полного снимка поискового корпуса. Мы используем temporal-style validation: оставляем только train-клики по `item_id`, которые существуют в `benchmark_items`. Поиск при этом всегда идёт по полному benchmark-корпусу из 189 212 объявлений. Это максимально близко к финальному inference, но labels остаются неполными: неизвестный клик не означает нерелевантность. Для сравнения recall разных retriever'ов это приемлемо, если manifest неизменен.

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
OUT = ROOT/'artifacts/validation'
OUT.mkdir(parents=True, exist_ok=True)
QUERY_COLUMNS = ['search_query','search_location_id','search_is_delivery_search','search_infm_params_text','search_category']
SEED = 42
N_VALIDATION = 2452

def stable_query_id(row):
    payload = '\x1f'.join(str(row[c]) for c in QUERY_COLUMNS)
    return hashlib.sha1(payload.encode('utf-8')).hexdigest()[:16]

def local_pool_bucket(values):
    return pd.cut(values, bins=[-1,0,99,499,1999,9999,np.inf], labels=['0','1-99','100-499','500-1999','2000-9999','10000+']).astype(str)

## Какие оси сложности контролируем

Основные квоты повторяют benchmark по совместной страте:

- `has_filters`: есть ли `search_infm_params_text`; фильтры меняют текст запроса и допустимое множество items;
- `local_pool_bucket`: сколько corpus items находится в `search_location_id`; маленький пул делает local recall проще, большой — сложнее.

Дополнительно сохраняем диагностические срезы `positive_bucket` (один или несколько выбранных items) и `query_frequency_bucket` (tail/head текст запроса). Они не могут повторять benchmark напрямую, поскольку у benchmark нет labels, но по ним обязательно считаются slice-метрики. Category и delivery рассматриваются отдельно: если train не содержит достаточной поддержки, стратификация не может её выдумать.

In [2]:
items = pd.read_parquet(ROOT/'dataset/benchmark_items.parquet', columns=['item_id','item_location_id'])
benchmark = pd.read_parquet(ROOT/'dataset/benchmark_queries.parquet')
train = pd.read_parquet(ROOT/'dataset/train.parquet', columns=QUERY_COLUMNS+['item_id'])
eligible_pairs = train[train.item_id.isin(set(items.item_id))].drop_duplicates()
positives = eligible_pairs.groupby(QUERY_COLUMNS, dropna=False).item_id.nunique().rename('positive_count').reset_index()
query_frequency = train.groupby('search_query').size()
location_pool = items.item_location_id.value_counts()

def add_strata(frame):
    result = frame.copy()
    result['has_filters'] = result.search_infm_params_text.ne('')
    result['local_pool_size'] = result.search_location_id.map(location_pool).fillna(0).astype(int)
    result['local_pool_bucket'] = local_pool_bucket(result.local_pool_size)
    result['primary_stratum'] = result.has_filters.astype(str)+'|'+result.local_pool_bucket
    return result

eligible = add_strata(positives)
eligible['query_frequency'] = eligible.search_query.map(query_frequency).astype(int)
eligible['query_frequency_bucket'] = pd.cut(eligible.query_frequency, bins=[0,1,5,20,np.inf], labels=['1','2-5','6-20','21+']).astype(str)
eligible['positive_bucket'] = pd.cut(eligible.positive_count, bins=[0,1,2,5,np.inf], labels=['1','2','3-5','6+']).astype(str)
benchmark_profile = add_strata(benchmark)

support = pd.DataFrame({
    'eligible': [len(eligible), eligible.search_category.eq(0).sum(), eligible.search_is_delivery_search.eq(1).sum(), eligible.has_filters.mean()],
    'benchmark': [len(benchmark), benchmark.search_category.eq(0).sum(), benchmark.search_is_delivery_search.eq(1).sum(), benchmark_profile.has_filters.mean()],
}, index=['rows','category=0','delivery=1','filter_rate'])
display(support)

,eligible,benchmark
rows,26556.000000,2452.000000
category=0,2.000000,222.000000
delivery=1,1.000000,0.000000
filter_rate,0.640985,0.369494


## Квотное семплирование

Для каждой `has_filters × local_pool_bucket` страты берём ровно столько запросов, сколько в benchmark. Семплирование детерминировано (`seed=42`). Основной benchmark не содержит delivery-запросов, поэтому delivery=1 не включается в aggregate metric. Две доступные размеченные точки category=0 включаются принудительно; больше взять невозможно — это честно фиксируем как blind spot.

In [3]:
quotas = benchmark_profile.primary_stratum.value_counts().to_dict()
available = eligible[eligible.search_is_delivery_search.eq(0)].copy()
parts = []
for stratum, quota in sorted(quotas.items()):
    candidates = available[available.primary_stratum.eq(stratum)]
    take = min(quota, len(candidates))
    parts.append(candidates.sample(n=take, random_state=SEED + sum(map(ord,stratum))))
manifest = pd.concat(parts).drop_duplicates(QUERY_COLUMNS).copy()

# Fill any quota deficits from the same filter state, preferring rows not yet selected.
if len(manifest) < N_VALIDATION:
    remaining = available.merge(manifest[QUERY_COLUMNS], on=QUERY_COLUMNS, how='left', indicator=True)
    remaining = remaining[remaining._merge.eq('left_only')].drop(columns='_merge')
    manifest = pd.concat([manifest, remaining.sample(N_VALIDATION-len(manifest), random_state=SEED)])

# Force all supported category=0 cases; replace matched primary-stratum rows to keep N fixed.
rare = available[available.search_category.eq(0)]
for _, row in rare.iterrows():
    same = (manifest.primary_stratum.eq(row.primary_stratum) & manifest.search_category.ne(0))
    if same.any() and not (manifest[QUERY_COLUMNS] == row[QUERY_COLUMNS].values).all(axis=1).any():
        manifest = manifest.drop(manifest[same].index[0])
        manifest = pd.concat([manifest, row.to_frame().T], ignore_index=True)

manifest['eval_query_id'] = manifest.apply(stable_query_id, axis=1)
manifest = manifest.drop_duplicates('eval_query_id').reset_index(drop=True)
assert len(manifest) == N_VALIDATION and manifest.eval_query_id.is_unique
labels = eligible_pairs.merge(manifest[QUERY_COLUMNS+['eval_query_id']], on=QUERY_COLUMNS, how='inner')[['eval_query_id','item_id']].drop_duplicates()
assert labels.eval_query_id.nunique() == len(manifest)
manifest.to_parquet(OUT/'manifest.parquet', index=False)
labels.to_parquet(OUT/'labels.parquet', index=False)
print(f'manifest={len(manifest):,}; labels={len(labels):,}; mean positives={len(labels)/len(manifest):.3f}')

manifest=2,452; labels=2,694; mean positives=1.099


## Проверка результата и границы применимости

Таблица ниже должна показать близкие доли primary strata между validation и benchmark. Диагностические strata нужны для последующего breakdown Recall@50. Даже идеальное совпадение не устраняет два ограничения: (1) train clicks — неполная релевантность; (2) для category=0 размеченной поддержки практически нет. Поэтому общий Recall используется для выбора retrieval-компонентов, а category=0 в финале обрабатывается как отсутствие category hard-filter.

In [4]:
comparison = pd.concat([
    manifest.primary_stratum.value_counts(normalize=True).rename('validation'),
    benchmark_profile.primary_stratum.value_counts(normalize=True).rename('benchmark'),
], axis=1).fillna(0).sort_index()
display(comparison.style.format('{:.2%}'))
display(pd.DataFrame({
    'query_frequency': manifest.query_frequency_bucket.value_counts(),
    'positive_count': manifest.positive_bucket.value_counts(),
}).fillna(0).astype(int))
print('category:', manifest.search_category.value_counts().to_dict())
print('delivery:', manifest.search_is_delivery_search.value_counts().to_dict())

,validation,benchmark
primary_stratum,,
False|0,11.58%,11.58%
False|1-99,1.18%,1.18%
False|100-499,5.22%,5.22%
False|10000+,14.56%,14.56%
False|2000-9999,13.78%,13.78%
False|500-1999,16.72%,16.72%
True|0,5.83%,5.83%
True|1-99,0.53%,0.53%
True|100-499,3.47%,3.47%


,query_frequency,positive_count
1,365,2283
2,0,128
2-5,463,0
21+,1198,0
3-5,0,39
6+,0,2
6-20,426,0


category: {114: 2450, 0: 2}
delivery: {0: 2452}
